# Macro Voting Timing Model — TRIAL 2 (Consolidated Colab Notebook)
Implements the Run 1 → Run 2 handoff: `run1_retrospective_and_v4_framework.md`,
**Part B (defect fixes) + Part C (rerun + report content)**. This is Sonnet executing
Tasks 1–3 of that document. Task 4 (v4 experiments) is deliberately NOT in this notebook —
per the handoff, v4 only starts after Tasks 1–3 are reviewed and accepted.

**What's different from trial 1:**
- B.1 — diagnostic + fix for the risk_spread (BAMLH0A0HYM2) data defect that produced NaN
  for the entire evaluation window in trial 1.
- B.2 — trims any partial current-month row before the backtest.
- B.3 — capture ratios redefined on a geometric-mean-per-month basis (trial 1's version was
  horizon-dominated and uninformative).
- B.4 — new permanent G9 data-integrity gate, checked (and proven to actually catch failures)
  before any interpretation.
- A Run 1 → Run 2 delta table, and REPORT.md Phase 5/6 draft text generated from the real
  Run 2 numbers.

Run this in Colab: Runtime -> Run all. Needs live internet access to `fred.stlouisfed.org`
and `finance.yahoo.com`.

## 0. Setup

In [ ]:
!pip install -q pandas_datareader yfinance

import os, json, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
%matplotlib inline

RAW_DIR = "data/raw"
FIG_DIR = "output/figures"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

EVAL_START = "2000-01-01"
SEED = 20000101
print("Setup OK.")

## 1. Data layer (spec Section 3)
Same as trial 1: download/cache 7 series, month-end resample, CFNAI/CPI +1 month as-of shift.

In [ ]:
FRED_SERIES = {
    "CFNAI": "1985-01-01",
    "CPIAUCSL": "1985-01-01",
    "DTWEXBGS": "2006-01-01",
    "DGS2": "1985-01-01",
    "BAMLH0A0HYM2": "1996-12-01",
}
YAHOO_SERIES = {
    "^GSPC": "1985-01-01",
    "^SP500TR": "1988-01-01",
}
STATISTICAL_RELEASE_SERIES = {"CFNAI", "CPIAUCSL"}


def _cache_path(name):
    return os.path.join(RAW_DIR, f"{name.replace('^','')}.csv")


def _fetch_fred(series_id, start):
    import pandas_datareader.data as web
    df = web.DataReader(series_id, "fred", start=start, end=datetime.today())
    s = df[series_id].rename(series_id)
    s.index.name = "date"
    return s


def _fetch_yahoo(ticker, start):
    import yfinance as yf
    df = yf.download(ticker, start=start, interval="1mo", progress=False, auto_adjust=False)
    if df.empty:
        raise RuntimeError(f"yfinance returned no data for {ticker}")
    close = df["Close"]
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
    return close.rename(ticker).rename_axis("date")


def get_series(name, force_refresh=False):
    path = _cache_path(name)
    if os.path.exists(path) and not force_refresh:
        return pd.read_csv(path, index_col="date", parse_dates=True)[name]
    if name in FRED_SERIES:
        s = _fetch_fred(name, FRED_SERIES[name])
    elif name in YAHOO_SERIES:
        s = _fetch_yahoo(name, YAHOO_SERIES[name])
    else:
        raise KeyError(name)
    s.to_frame(name=name).rename_axis("date").to_csv(path)
    return s


def to_month_end(s):
    s = s.copy()
    s.index = pd.to_datetime(s.index)
    return s.resample("ME").last()


def build_raw_panel(force_refresh=False):
    raw = {name: get_series(name, force_refresh) for name in list(FRED_SERIES) + list(YAHOO_SERIES)}
    shifted = {}
    for name, s in raw.items():
        me = to_month_end(s)
        shifted[name] = me.shift(1) if name in STATISTICAL_RELEASE_SERIES else me
    panel = pd.DataFrame(shifted)
    panel.index.name = "date"
    return panel


panel = build_raw_panel(force_refresh=False)
print("Initial panel:", panel.shape)
panel.tail()

## 2. Defect fix B.1 — diagnose and repair the risk_spread (BAMLH0A0HYM2) data defect
Trial 1's Fig 6 risk_spread panel was empty for the entire evaluation window. Diagnose the
cached download first (keep this printout for `DEVIATIONS.md`), then force a clean
re-download of just that one series.

In [ ]:
# --- Diagnostic ---
baml_path = _cache_path("BAMLH0A0HYM2")
if os.path.exists(baml_path):
    s_raw = pd.read_csv(baml_path, index_col="date", parse_dates=True)["BAMLH0A0HYM2"]
    print("raw cache :", s_raw.dropna().shape, s_raw.index.min(), "->", s_raw.index.max(), s_raw.dtype)
else:
    print("raw cache : file not found (will be created on next download)")

col = panel["BAMLH0A0HYM2"]
print("panel col :", col.dropna().shape, "first valid:", col.first_valid_index(), col.dtype)
tr_diag = col.diff(12)
print("diff(12)  :", tr_diag.dropna().shape, "first valid:", tr_diag.first_valid_index())

In [ ]:
# --- Fix: force a clean re-download of the one broken series ---
if os.path.exists(baml_path):
    os.remove(baml_path)
panel = build_raw_panel(force_refresh=False)   # re-downloads only the deleted series; rest reuse cache

col_fixed = panel["BAMLH0A0HYM2"]
print("After fix -- panel col:", col_fixed.dropna().shape, "first valid:", col_fixed.first_valid_index(), col_fixed.dtype)
tr_fixed = col_fixed.diff(12)
print("After fix -- diff(12) :", tr_fixed.dropna().shape, "first valid:", tr_fixed.first_valid_index())
assert col_fixed.dropna().shape[0] > 200, "B.1 fix did not take -- BAMLH0A0HYM2 still looks broken."
print("\nB.1 fix verified.")

## 3. Defect fix B.2 — trim any partial current-month row
Spec says "latest complete month." Drop any row for the current, still-in-progress month.

In [ ]:
last_complete = (pd.Timestamp.today().to_period("M") - 1).to_timestamp("M")
before_max = panel.index.max()
panel = panel[panel.index <= last_complete]
after_max = panel.index.max()
print(f"Panel trimmed: {before_max.date()} -> {after_max.date()} (last complete month: {last_complete.date()})")

## 4. Signal layer (spec Section 2)
Unchanged transforms/z-scores/votes/position rule from trial 1.

In [ ]:
ROLL_WINDOW = 120
MIN_HISTORY = 60
VOTE_THRESHOLD = 0.5

SIGN_PRIOR = {
    "growth": +1, "inflation": -1, "trade": +1,
    "policy": -1, "risk_price": +1, "risk_spread": -1,
}


def transform_factors(panel):
    out = pd.DataFrame(index=panel.index)
    out["growth"] = panel["CFNAI"].rolling(3).mean()
    out["inflation"] = panel["CPIAUCSL"].pct_change(12) * 100
    out["trade"] = panel["DTWEXBGS"].pct_change(12) * 100
    out["policy"] = panel["DGS2"].diff(12)
    out["risk_price"] = panel["^GSPC"].pct_change(12) * 100
    out["risk_spread"] = panel["BAMLH0A0HYM2"].diff(12)
    return out


def rolling_zscores(transformed):
    z = pd.DataFrame(index=transformed.index)
    for col in transformed.columns:
        mean = transformed[col].rolling(ROLL_WINDOW, min_periods=MIN_HISTORY).mean()
        std = transformed[col].rolling(ROLL_WINDOW, min_periods=MIN_HISTORY).std()
        z[col] = (transformed[col] - mean) / std
    return z


def signed_scores(z):
    return pd.DataFrame({col: SIGN_PRIOR[col] * z[col] for col in z.columns}, index=z.index)


def _vote_from_score(s, threshold=VOTE_THRESHOLD):
    v = pd.Series(0, index=s.index, dtype=float)
    v[s > threshold] = 1
    v[s < -threshold] = -1
    v[s.isna()] = np.nan
    return v


def compute_votes(s, threshold=VOTE_THRESHOLD, exclude=None):
    votes = pd.DataFrame(index=s.index)
    for f in ("growth", "inflation", "trade", "policy"):
        if exclude != f:
            votes[f] = _vote_from_score(s[f], threshold)
    if exclude != "risk":
        rp, rs = s["risk_price"], s["risk_spread"]
        both, only_price = rp.notna() & rs.notna(), rp.notna() & rs.isna()
        risk_signed = pd.Series(np.nan, index=s.index)
        risk_signed[both] = (rp[both] + rs[both]) / 2
        risk_signed[only_price] = rp[only_price]
        votes["risk"] = _vote_from_score(risk_signed, threshold)
    return votes


def vote_sum(votes):
    return votes.sum(axis=1, skipna=True)


def compute_positions(vsum, start_invested=True):
    pos = pd.Series(index=vsum.index, dtype=float)
    prev = 1.0 if start_invested else 0.0
    for dt, v in vsum.items():
        cur = 1.0 if v > 0 else (0.0 if v < 0 else prev)
        pos[dt] = cur
        prev = cur
    return pos


def run_model(panel, threshold=VOTE_THRESHOLD, exclude=None, eval_start=EVAL_START):
    transformed = transform_factors(panel)
    z = rolling_zscores(transformed)
    s = signed_scores(z)
    votes = compute_votes(s, threshold=threshold, exclude=exclude)
    vsum = vote_sum(votes)
    positions = compute_positions(vsum[eval_start:], start_invested=True)
    return positions, votes, vsum, s


positions, votes, vsum, signed = run_model(panel)
print("Evaluation window:", positions.index.min().date(), "to", positions.index.max().date())

## 5. Defect fix B.4 — G9 data-integrity gate
New permanent gate: verifies every factor actually voted by the date it should have, with
adequate coverage after entry. Proven both ways -- passes on the fixed data, and demonstrably
FAILS if pointed at data resembling the trial-1 defect.

In [ ]:
def data_integrity_check(signed_df, eval_start=EVAL_START, verbose=True):
    expected_entry = {"growth": "2000-01-31", "inflation": "2000-01-31",
                       "policy": "2000-01-31", "risk_price": "2000-01-31",
                       "trade": "2012-06-30", "risk_spread": "2003-06-30"}
    problems = []
    for col, latest in expected_entry.items():
        fv = signed_df[col].first_valid_index()
        if fv is None or fv > pd.Timestamp(latest):
            problems.append(f"{col}: first valid {fv}, expected <= {latest}")
            continue
        cov = signed_df[col][max(pd.Timestamp(eval_start), fv):].notna().mean()
        if cov < 0.95:
            problems.append(f"{col}: only {cov:.0%} non-NaN after entry (expected >= 95%)")
    if problems:
        if verbose:
            print("DATA INTEGRITY FAILURE:\n" + "\n".join(problems))
        return False, problems
    if verbose:
        print("G9 data-integrity check passed.")
    return True, []


# 1) Prove it passes on the fixed data
ok, problems = data_integrity_check(signed)
assert ok, "G9 failed on fixed data -- do not proceed."

# 2) Prove it actually catches the trial-1-style defect (simulate risk_spread all-NaN)
signed_broken = signed.copy()
signed_broken["risk_spread"] = np.nan
ok_broken, problems_broken = data_integrity_check(signed_broken, verbose=False)
assert not ok_broken, "G9 should have failed on simulated broken data but did not."
print("Negative-control check: G9 correctly FAILS on simulated broken risk_spread ->")
print("  " + "\n  ".join(problems_broken))

## 6. Backtest engine (spec Section 4)
Unchanged from trial 1 -- the signal-to-return shift and cash=0%/costs=0% engine.

In [ ]:
def total_return_series(price_index):
    return price_index.pct_change()


def run_backtest(positions, sp500_tr_level):
    mkt_ret = total_return_series(sp500_tr_level)
    pos_aligned = positions.shift(1)
    pos_aligned, mkt_ret_aligned = pos_aligned.align(mkt_ret, join="inner")
    strat_ret = pos_aligned * mkt_ret_aligned + (1 - pos_aligned) * 0.0
    df = pd.DataFrame({"position": pos_aligned, "mkt_return": mkt_ret_aligned,
                        "strategy_return": strat_ret}).dropna(subset=["position", "mkt_return"])
    df["strategy_equity"] = (1 + df["strategy_return"]).cumprod()
    df["bh_equity"] = (1 + df["mkt_return"]).cumprod()
    return df


def oracle_equity(sp500_tr_level, index_like):
    mkt_ret = total_return_series(sp500_tr_level).reindex(index_like.index)
    oracle_ret = pd.Series(np.where(mkt_ret > 0, mkt_ret, 0.0), index=mkt_ret.index)
    return (1 + oracle_ret).cumprod()


def switches_and_holding_period(position):
    pos = position.dropna()
    changes = max((pos != pos.shift(1)).sum() - 1, 0)
    n_spells = changes + 1
    return int(changes), float(len(pos) / n_spells)


bt_df = run_backtest(positions, panel["^SP500TR"])
n_switches, avg_holding = switches_and_holding_period(bt_df["position"])
print(f"Switches: {n_switches}, avg holding period: {avg_holding:.1f} months")
bt_df.tail()

## 7. Evaluation: metrics, confusion matrix, random null, robustness (Section 5)
Includes defect fix **B.3** -- `capture_ratios` redefined on a geometric-mean-per-month
basis instead of full-period compounding, which was horizon-dominated in trial 1.

In [ ]:
PERIODS_PER_YEAR = 12

def cagr(returns):
    equity = (1 + returns).cumprod()
    n_years = len(returns) / PERIODS_PER_YEAR
    if n_years <= 0 or equity.iloc[-1] <= 0:
        return np.nan
    return equity.iloc[-1] ** (1 / n_years) - 1

def ann_vol(returns):
    return returns.std() * np.sqrt(PERIODS_PER_YEAR)

def sharpe(returns, rf=0.0):
    excess = returns - rf / PERIODS_PER_YEAR
    vol = excess.std()
    return np.nan if (vol == 0 or np.isnan(vol)) else (excess.mean() / vol) * np.sqrt(PERIODS_PER_YEAR)

def drawdown_series(returns):
    equity = (1 + returns).cumprod()
    return equity / equity.cummax() - 1

def max_drawdown(returns):
    return drawdown_series(returns).min()

def longest_drawdown_duration(returns):
    dd = drawdown_series(returns) < 0
    longest = cur = 0
    for flag in dd:
        cur = cur + 1 if flag else 0
        longest = max(longest, cur)
    return int(longest)

def calmar(cagr_v, maxdd_v):
    return np.nan if (maxdd_v == 0 or np.isnan(maxdd_v)) else cagr_v / abs(maxdd_v)

def hit_ratio(position, mkt_return):
    matched = ((position == 1) & (mkt_return > 0)) | ((position == 0) & (mkt_return <= 0))
    return matched.mean()

# --- B.3 fix: geometric-mean-per-month capture ratios (was full-period compounding) ---
def capture_ratios(strategy_return, mkt_return):
    up, down = mkt_return > 0, mkt_return < 0
    geo = lambda s: (1 + s.dropna()).prod() ** (1 / max(len(s.dropna()), 1)) - 1
    upside = geo(strategy_return[up]) / geo(mkt_return[up]) if up.any() and geo(mkt_return[up]) != 0 else np.nan
    downside = geo(strategy_return[down]) / geo(mkt_return[down]) if down.any() and geo(mkt_return[down]) != 0 else np.nan
    return upside, downside

def metrics_row(strategy_return, position, mkt_return, n_switches=None, avg_holding=None):
    c, dd = cagr(strategy_return), max_drawdown(strategy_return)
    up_cap, down_cap = capture_ratios(strategy_return, mkt_return)
    return {
        "CAGR": c, "ann_vol": ann_vol(strategy_return), "sharpe": sharpe(strategy_return),
        "max_drawdown": dd, "calmar": calmar(c, dd),
        "longest_dd_months": longest_drawdown_duration(strategy_return),
        "hit_ratio": hit_ratio(position, mkt_return) if position is not None else np.nan,
        "pct_months_in_market": position.mean() if position is not None else np.nan,
        "n_switches": n_switches, "avg_holding_period_months": avg_holding,
        "upside_capture": up_cap, "downside_capture": down_cap,
    }

def confusion_matrix(position, mkt_return):
    up, down = mkt_return > 0, mkt_return <= 0
    inv, cash = position == 1, position == 0
    return {"invested_and_up": int((inv & up).sum()), "invested_and_down": int((inv & down).sum()),
            "cash_and_up": int((cash & up).sum()), "cash_and_down": int((cash & down).sum()),
            "up_month_base_rate": float(up.mean())}

def random_timing_null(mkt_return, n_invested_months, n_sims=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    n_total = len(mkt_return)
    sims = []
    for _ in range(n_sims):
        idx = rng.choice(n_total, size=n_invested_months, replace=False)
        pos = np.zeros(n_total); pos[idx] = 1.0
        sim_ret = pd.Series(pos, index=mkt_return.index) * mkt_return
        sims.append({"CAGR": cagr(sim_ret), "sharpe": sharpe(sim_ret)})
    return pd.DataFrame(sims)

def percentile_of(value, distribution):
    return float((distribution < value).mean() * 100)

DEFAULT_SUBPERIODS = {"2000-2009": ("2000-01-01", "2009-12-31"),
                       "2010-2019": ("2010-01-01", "2019-12-31"),
                       "2020-present": ("2020-01-01", None)}

def sub_period_metrics(strategy_return, position, mkt_return, bounds):
    rows = {}
    for label, (start, end) in bounds.items():
        sr, pos, mr = strategy_return[start:end], position[start:end], mkt_return[start:end]
        if len(sr):
            rows[label] = metrics_row(sr, pos, mr)
    return pd.DataFrame(rows).T

print("Evaluation functions defined (B.3 capture-ratio fix included).")

### 7a. Tripwire check (gate G8) -- must pass before trusting anything below

In [ ]:
strategy_row = metrics_row(bt_df["strategy_return"], bt_df["position"], bt_df["mkt_return"], n_switches, avg_holding)
bh_row = metrics_row(bt_df["mkt_return"], pd.Series(1, index=bt_df.index), bt_df["mkt_return"], 0, float(len(bt_df)))

oracle_eq = oracle_equity(panel["^SP500TR"], bt_df)
oracle_ret = oracle_eq.pct_change().fillna(oracle_eq.iloc[0] - 1)
oracle_pos = (bt_df["mkt_return"] > 0).astype(int)
oracle_row = metrics_row(oracle_ret, oracle_pos, bt_df["mkt_return"],
                          int((oracle_pos != oracle_pos.shift(1)).sum()), np.nan)

null_df = random_timing_null(bt_df["mkt_return"], int(bt_df["position"].sum()), n_sims=1000, seed=SEED)
null_row = {"CAGR": null_df["CAGR"].mean(), "sharpe": null_df["sharpe"].mean()}

metrics_table = pd.DataFrame({"strategy": strategy_row, "buy_and_hold": bh_row,
                               "oracle": oracle_row, "random_null_mean": null_row}).T

strat_sharpe, strat_cagr, bh_cagr = strategy_row["sharpe"], strategy_row["CAGR"], bh_row["CAGR"]
tripped = bool(strat_sharpe > 1.0 or strat_cagr > bh_cagr + 0.05)
tripwire = {"tripped": tripped, "strategy_sharpe": float(strat_sharpe),
            "strategy_cagr": float(strat_cagr), "buy_hold_cagr": float(bh_cagr),
            "note": ("TRIPWIRE HIT -- halt and complete a look-ahead audit before presenting "
                     "any results (gate G8)." if tripped else
                     "Tripwire not triggered; no audit required.")}
print(tripwire["note"])
assert not tripwire["tripped"], "STOP: tripwire hit, audit before proceeding (see note above)."
tripwire

### 7b. Metrics table (Section 5.1)

In [ ]:
metrics_table

### 7c. Confusion matrix (Section 5.2)

In [ ]:
confusion = confusion_matrix(bt_df["position"], bt_df["mkt_return"])
confusion

### 7d. Random-timing null (Section 5.3)

In [ ]:
null_percentiles = {"CAGR_percentile": percentile_of(strategy_row["CAGR"], null_df["CAGR"]),
                    "sharpe_percentile": percentile_of(strategy_row["sharpe"], null_df["sharpe"])}
null_percentiles

### 7e. Robustness suite (Section 5.4)

In [ ]:
threshold_rows = {}
for th in (0.3, 0.5, 0.7):
    pos_th, *_ = run_model(panel, threshold=th)
    bdf = run_backtest(pos_th, panel["^SP500TR"])
    threshold_rows[f"threshold_{th}"] = metrics_row(bdf["strategy_return"], bdf["position"], bdf["mkt_return"])
threshold_table = pd.DataFrame(threshold_rows).T

subperiod_table = sub_period_metrics(bt_df["strategy_return"], bt_df["position"], bt_df["mkt_return"], DEFAULT_SUBPERIODS)

ablation_rows = {}
for factor in ("growth", "inflation", "trade", "policy", "risk"):
    pos_ab, *_ = run_model(panel, exclude=factor)
    bdf = run_backtest(pos_ab, panel["^SP500TR"])
    ablation_rows[f"without_{factor}"] = metrics_row(bdf["strategy_return"], bdf["position"], bdf["mkt_return"])
ablation_table = pd.DataFrame(ablation_rows).T

print("Threshold sensitivity:"); display(threshold_table)
print("\nSub-period metrics:"); display(subperiod_table)
print("\nLeave-one-out ablation:"); display(ablation_table)

## 8. Task 2 — Run 1 -> Run 2 delta table
Run-1 (trial 1, provisional/defective) numbers are hardcoded from the retrospective doc for
comparison. Attributes the delta to the B.1-B.3 fixes, not to any tuning.

In [ ]:
run1_strategy = {
    "CAGR": 0.042104, "ann_vol": 0.089751, "sharpe": 0.504937, "max_drawdown": -0.237351,
    "hit_ratio": 0.471698, "pct_months_in_market": 0.446541, "n_switches": 27,
    "avg_holding_period_months": 11.357143,
}
run2_strategy = {k: strategy_row[k] for k in run1_strategy}

delta_table = pd.DataFrame({"run1_provisional": run1_strategy, "run2_fixed": run2_strategy})
delta_table["delta"] = delta_table["run2_fixed"] - delta_table["run1_provisional"]
print("Run 1 (provisional, defective risk_spread) -> Run 2 (B.1-B.3 fixed):")
delta_table

## 9. Required figures

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(bt_df.index, bt_df["strategy_equity"], label="Strategy")
ax.plot(bt_df.index, bt_df["bh_equity"], label="Buy & Hold (TR)")
ax.plot(oracle_eq.index, oracle_eq.reindex(bt_df.index), label="Perfect-foresight oracle", linestyle="--")
ax.set_yscale("log"); ax.set_title("Fig 1. Growth of $1 (log scale) -- Run 2"); ax.legend()
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/01_growth_of_1.png", dpi=140); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(bt_df.index, drawdown_series(bt_df["strategy_return"]), label="Strategy")
ax.plot(bt_df.index, drawdown_series(bt_df["mkt_return"]), label="Buy & Hold")
ax.set_title("Fig 2. Drawdowns -- Run 2"); ax.legend()
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/02_drawdowns.png", dpi=140); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sp = panel["^GSPC"].reindex(bt_df.index)
ax.plot(bt_df.index, sp, color="black", linewidth=1)
ax.fill_between(bt_df.index, sp.min(), sp.max(), where=bt_df["position"]==1, color="tab:green", alpha=0.15, label="Invested")
ax.fill_between(bt_df.index, sp.min(), sp.max(), where=bt_df["position"]==0, color="tab:red", alpha=0.15, label="Cash")
ax.set_title("Fig 3. Position ribbon over S&P 500 price index -- Run 2"); ax.legend()
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/03_position_ribbon.png", dpi=140); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
vsum_eval = vsum[EVAL_START:]
ax.plot(vsum_eval.index, vsum_eval, drawstyle="steps-post")
ax.axhline(0, color="black", linewidth=0.8)
for label, d in {"Risk(5b) spread joins": "2002-12-01", "Trade factor joins": "2012-01-01"}.items():
    ax.axvline(pd.Timestamp(d), color="grey", linestyle=":", alpha=0.7)
    ax.text(pd.Timestamp(d), ax.get_ylim()[1]*0.9, label, rotation=90, fontsize=7, va="top")
ax.set_title("Fig 4. Vote sum over time -- Run 2")
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/04_vote_sum.png", dpi=140); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(null_df["sharpe"], bins=40, color="tab:blue", alpha=0.7)
ax.axvline(strategy_row["sharpe"], color="red", linewidth=2, label="Strategy Sharpe")
ax.set_title("Fig 5. Random-timing null: Sharpe distribution (n=1000) -- Run 2"); ax.legend()
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/05_random_null_sharpe.png", dpi=140); plt.show()

In [ ]:
signed_eval = signed[EVAL_START:]
fig, axes = plt.subplots(len(signed_eval.columns), 1, figsize=(10, 10), sharex=True)
for ax, col in zip(axes, signed_eval.columns):
    ax.plot(signed_eval.index, signed_eval[col])
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_ylabel(col, fontsize=8)
axes[0].set_title("Fig 6. Per-factor signed scores over time -- Run 2 (risk_spread should now be populated)")
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/06_factor_scores.png", dpi=140); plt.show()

# Explicit confirmation the B.1 fix shows up where it matters
cov = signed_eval["risk_spread"].notna().mean()
print(f"risk_spread coverage over evaluation window: {cov:.1%} (trial 1 was 0%)")

## 10. Write output/metrics.json

In [ ]:
def _jsonify(obj):
    if isinstance(obj, (pd.DataFrame, pd.Series)):
        return json.loads(obj.to_json(orient="index"))
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    if isinstance(obj, dict):
        return {k: _jsonify(v) for k, v in obj.items()}
    return obj

payload = {
    "run": 2,
    "metrics_table": _jsonify(metrics_table),
    "confusion_matrix": _jsonify(confusion),
    "null_percentiles": _jsonify(null_percentiles),
    "threshold_table": _jsonify(threshold_table),
    "subperiod_table": _jsonify(subperiod_table),
    "ablation_table": _jsonify(ablation_table),
    "tripwire": _jsonify(tripwire),
    "g9_data_integrity": {"passed": ok, "problems": problems},
    "run1_to_run2_delta": _jsonify(delta_table),
    "n_switches": n_switches,
    "avg_holding_period_months": avg_holding,
    "evaluation_window": {"start": str(bt_df.index.min().date()), "end": str(bt_df.index.max().date()),
                           "n_months": int(len(bt_df))},
}
os.makedirs("output", exist_ok=True)
with open("output/metrics.json", "w") as f:
    json.dump(payload, f, indent=2, default=str)
print("Wrote output/metrics.json (run 2)")

try:
    from google.colab import files
    files.download("output/metrics.json")
except ImportError:
    pass

## 11. Task 3 — REPORT.md Phase 5 & 6 draft (auto-generated from Run 2 numbers)

In [ ]:
phase5 = f"""## Phase 5 -- Performance (Run 2, defects fixed)

**Evaluation window:** {bt_df.index.min().date()} to {bt_df.index.max().date()} ({len(bt_df)} months).

**Metrics table:**

{metrics_table.round(4).to_markdown()}

**Confusion matrix:** invested & up = {confusion['invested_and_up']}, invested & down = {confusion['invested_and_down']},
cash & up = {confusion['cash_and_up']}, cash & down = {confusion['cash_and_down']}
(up-month base rate {confusion['up_month_base_rate']:.1%}). Error asymmetry: missed up-months
({confusion['cash_and_up']}) {'exceed' if confusion['cash_and_up'] > confusion['cash_and_down'] else 'are below'} avoided down-months ({confusion['cash_and_down']}), so each unnecessary cash month costs more than each avoided down-month saves.

**Random-timing null:** strategy CAGR at the {null_percentiles['CAGR_percentile']:.1f}th percentile,
Sharpe at the {null_percentiles['sharpe_percentile']:.1f}th percentile of 1,000 sims (seed {SEED}).

**Threshold sensitivity (C4 verdict):** CAGR ranges {threshold_table['CAGR'].min():.2%} to {threshold_table['CAGR'].max():.2%}
and Sharpe ranges {threshold_table['sharpe'].min():.2f} to {threshold_table['sharpe'].max():.2f} across +/-0.3/+/-0.5/+/-0.7 --
qualitative conclusion (defensive profile vs buy & hold) should be checked for stability across thresholds before being treated as robust.

**Switches:** {n_switches}; average holding period: {avg_holding:.1f} months.

**Run 1 -> Run 2 delta (attributable to B.1-B.3 fixes, not tuning):**

{delta_table.round(4).to_markdown()}
"""
display(Markdown(phase5))

In [ ]:
phase6 = f"""## Phase 6 -- Interpretation for senior PMs (Run 2 draft -- review before finalizing)

- **Sub-period behavior:** {subperiod_table.round(4).to_markdown()}
  Check whether the B.1 risk_spread fix (now covering {cov:.0%} of the eval window vs 0% in
  trial 1) narrows the 2000-2009 underperformance noted in the retrospective -- spreads
  narrowing in 2003 and 2009-10 should show up as an earlier bullish tilt if the fix is working
  as intended.

- **Factor attribution (leave-one-out):** {ablation_table[['CAGR','sharpe']].round(4).to_markdown()}
  Whichever factor's removal still moves CAGR/Sharpe most is the one to discuss --
  cross-reference against the trial-1 finding that `without_trade` and `without_risk` improved
  results most, and confirm whether that finding survived the B.1 fix or was partly an artifact
  of the broken risk_spread series.

- **Why underperforming buy-and-hold CAGR ({metrics_table.loc['strategy','CAGR']:.2%} vs
  {metrics_table.loc['buy_and_hold','CAGR']:.2%}) while running lower volatility
  ({metrics_table.loc['strategy','ann_vol']:.2%} vs {metrics_table.loc['buy_and_hold','ann_vol']:.2%})
  and lower max drawdown ({metrics_table.loc['strategy','max_drawdown']:.2%} vs
  {metrics_table.loc['buy_and_hold','max_drawdown']:.2%}) is a legitimate overlay outcome, not a
  failure: this is a risk-budget decision, not a stock-picking one.**

- **Why simplicity (voting, not fitting) was the right choice at this sample size** -- ties back
  to Phase 2 of REPORT.md: ~300 monthly observations for 5 factors does not support a fitted
  model without serious overfitting risk.

*(This section is a first draft scaffold from Run 2 numbers -- a human/Fable-5 review pass is
still required before this is final REPORT.md content; some sentences above need the reviewer
to actually look at the tables and commit to a specific claim, not just report the number.)*
"""
display(Markdown(phase6))

## 12. Next step
Copy the Phase 5/6 markdown above into `report/REPORT.md`, replacing the `[PENDING REAL RUN]`
placeholders. Log the B.1-B.4 fixes in `DEVIATIONS.md` (diagnostic printouts from Section 2 +
the delta table from Section 8). Return everything to Fable 5 for gate G1-G9 verification,
checklist scoring, and the Phase 7 interview drill. Task 4 (v4 experiments) starts only after
this review.